In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "main" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "finetuning_results.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all

In [ ]:
analysis_dir = repo_dir / 'analysis'
config_dir = analysis_dir / 'curve_fitting/configs/finetuning'
results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results'
# results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results_finetuning'

In [ ]:
finetuning_datasets = [
    'tvsd',
    'things_fmri',
    'nsd',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
]

benchmarks = [
    'bs_fz',
    'bs_mh',
    'tvsd',
    'things_fmri',
    'nsd',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
    'benchmark_average',
]

METRIC = 'pearsonr_nc'

In [ ]:
all_configs = {}

for f_ds in finetuning_datasets:
    for bench in benchmarks:
        yaml_config = config_dir / f'{f_ds}/{bench}.yaml'
        all_configs[f"f_{f_ds}-b_{bench}"] = load_yaml(yaml_config)

# yaml_config = config_dir / f'architecture_average/benchmark_average.yaml'
# all_configs[f"architecture_average-benchmark_average"] = load_yaml(yaml_config)

In [ ]:
L_fit_dict = {key: config['fitting_parameters']['loss_function'] for key, config in all_configs.items()}
L_viz_dict = {key: config['visualization']['loss_function'] for key, config in all_configs.items()}
x_scale_dict = {key: float(config['fitting_parameters']['X_scaler']) for key, config in all_configs.items()}

In [ ]:
all_df = {
    name: apply_filters(df_all, config.get('data_filters', {}))
    for name, config in all_configs.items()
}

In [ ]:
optimized_params_dict = {}
opt_params_boot_dict = {}

for cfg in all_configs.keys():
    try:
        with open(results_dir / f'finetuning-{cfg}' / 'results.pkl', 'rb') as f:
            results = pickle.load(f)

        L_fit = L_fit_dict[cfg]
        L_viz = L_viz_dict[cfg]
        optimized_params_dict[cfg] = convert_loss_parameters(results['optimized_parameters'], L_fit, L_viz)

        # Convert bootstrapped parameters
        opt_params_boot = results['optimized_parameters_bootstrapped']
        opt_params_boot_dict[cfg] = convert_loss_parameters_batch(
            params=opt_params_boot,
            src_loss=L_fit,
            dst_loss=L_viz
        )
    except Exception as e:
        print(f"Could not load results for config {cfg}: {e}")

In [ ]:
x_extend = 15
X_str = r'$$\tilde{D}$$'
linewidth = 3.0
alpha_scatter = 0.2
alpha_ci = 0.2
alpha_fit = 1.0
fig_multiplier = 0.75
fig_multiplier = 0.6
# fig_multiplier = 2.0
# fig_multiplier = 1.0
figsize = (10, 8)
figsize = (10, 7)
figsize = (10, 7)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)

bench = 'benchmark_average'
ax = axes
for j, f_ds in enumerate(finetuning_datasets):
    cfg = f"f_{f_ds}-b_{bench}"
    
    if cfg not in optimized_params_dict:
        print(f"Skipping config {cfg} as no optimized parameters found.")
        continue

    df_plot = all_df[cfg]
    optimized_params = optimized_params_dict[cfg]
    opt_params_boot = opt_params_boot_dict[cfg]
    L = LOSS_FUNCTIONS[L_viz_dict[cfg]]
    x_scaler = x_scale_dict[cfg]
    X = df_plot.fitting_samples.values


    color = BENCHMARK_COLORS.get(f_ds)
    # sns.lineplot(data=df_plot, x='fitting_samples', y='pearsonr_nc', ax=ax, color=color, label=f'{benchmark_names[f_ds]}')
    sns.lineplot(
        data=df_plot,
        x='fitting_samples',
        y='pearsonr_nc', 
        marker='',
        # linestyles='-',
        ax=ax, 
        color=color, 
        label=f'{BENCHMARK_NAME_MAPPING[f_ds]}'
    )
    sns.scatterplot(
        data=df_plot,
        x='fitting_samples',
        y='pearsonr_nc', 
        marker='o',
        linestyle='-',
        ax=ax, 
        color=color, 
        alpha=0.4,
        # label=f'{benchmark_names[f_ds]}'
        legend=False
    )


ax.set_xscale('log')
ax.set_xlabel('Neural Samples', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Alignment', fontsize=16, fontweight='bold')
ax.set_title('Scaling Neural Fine-Tuning', fontsize=20, fontweight='bold')
ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.528, 0.530, 0.532, 0.534, 0.536, 0.538], precision=3)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, fontsize=10, loc='upper left')
print(labels)

ax.spines[['right', 'top']].set_visible(False)

plt.tight_layout()


figures_dir = save_dir
fig_name = 'fig4-finetuning-improvements-scaling'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)